In [1]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict , Annotated,Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()
llm=ChatGroq(model="llama-3.3-70b-versatile")


In [5]:
class JokeState(TypedDict):
    topic:str
    joke: str
    explanation:str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}


In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.',
 'explanation': 'This joke is a play on words, using a pun to create humor. The setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to the pizza\'s emotional state.\n\nThe punchline "Because it was feeling a little crusty" is where the wordplay occurs. In this context, "crusty" has a double meaning:\n\n1. A pizza has a crust, which is the outer layer of the bread. So, "crusty" can literally refer to the pizza\'s crust.\n2. In everyday language, "feeling crusty" is an idiomatic expression that means being irritable, grumpy, or in a bad mood. This phrase is often used to describe someone who is being short-tempered or cranky.\n\nThe joke relies on the unexpected twist of using the word "crusty" to describe both the pizza\'s physical characteristic (its crust) and its emotional state (being in a bad mood). The humor comes fro

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a pun to create humor. The setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to the pizza\'s emotional state.\n\nThe punchline "Because it was feeling a little crusty" is where the wordplay occurs. In this context, "crusty" has a double meaning:\n\n1. A pizza has a crust, which is the outer layer of the bread. So, "crusty" can literally refer to the pizza\'s crust.\n2. In everyday language, "feeling crusty" is an idiomatic expression that means being irritable, grumpy, or in a bad mood. This phrase is often used to describe someone who is being short-tempered or cranky.\n\nThe joke relies on the unexpected twist of using the word "crusty" to describe both the pizza\'s physical characteristic (its crust) and its emotional state (being in a bad mood). 

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a pun to create humor. The setup for the joke is "Why was the pizza in a bad mood?" which primes the listener to expect a reason related to the pizza\'s emotional state.\n\nThe punchline "Because it was feeling a little crusty" is where the wordplay occurs. In this context, "crusty" has a double meaning:\n\n1. A pizza has a crust, which is the outer layer of the bread. So, "crusty" can literally refer to the pizza\'s crust.\n2. In everyday language, "feeling crusty" is an idiomatic expression that means being irritable, grumpy, or in a bad mood. This phrase is often used to describe someone who is being short-tempered or cranky.\n\nThe joke relies on the unexpected twist of using the word "crusty" to describe both the pizza\'s physical characteristic (its crust) and its emotional state (being in a bad mood).

In [15]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment.',
 'explanation': 'The joke "Why did the spaghetti refuse to get married? Because it was afraid of getting tangled up in a lifelong commitment" is a play on words. It\'s a pun that combines the literal meaning of "tangled up" (as in, the physical state of being entwined or knotted) with the figurative meaning of being emotionally or financially entangled in a relationship.\n\nSpaghetti, being a long, thin, and flexible type of pasta, is prone to getting tangled or knotted when cooked or handled. The joke takes this physical characteristic and applies it to the concept of marriage, which is often referred to as a "lifelong commitment." The punchline suggests that the spaghetti is hesitant to get married because it\'s afraid of becoming emotionally or financially "tangled up" in the relationship, much like its physical tendency to become knott

In [17]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment.', 'explanation': 'The joke "Why did the spaghetti refuse to get married? Because it was afraid of getting tangled up in a lifelong commitment" is a play on words. It\'s a pun that combines the literal meaning of "tangled up" (as in, the physical state of being entwined or knotted) with the figurative meaning of being emotionally or financially entangled in a relationship.\n\nSpaghetti, being a long, thin, and flexible type of pasta, is prone to getting tangled or knotted when cooked or handled. The joke takes this physical characteristic and applies it to the concept of marriage, which is often referred to as a "lifelong commitment." The punchline suggests that the spaghetti is hesitant to get married because it\'s afraid of becoming emotionally or financially "tangled up" in the relationship, much like its physical tende

In [22]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12c5e7-550c-6dc9-8001-fc70dfe01edc'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-30T17:32:43.197383+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12c5b6-dd81-68f7-8000-11b5d143e3da'}}, tasks=(PregelTask(id='2c5448ef-7e81-7504-c258-e040a6d39eb2', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married. \n\nBecause it was afraid of getting tangled up in a relationship.', 'explanation': 'The joke "Why did the spaghetti refuse to get married. Because it was afraid of getting tangled up in a relationship" is a play on words. \n\nThe phrase "tangled up in a relationship" is a common idio

In [19]:
# Time Travel
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f12c5b6-dd81-68f7-8000-11b5d143e3da"}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f12c5b6-dd81-68f7-8000-11b5d143e3da'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-30T17:11:02.172082+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12c5b6-dd4b-66fc-bfff-fb8191b3cffb'}}, tasks=(PregelTask(id='66a60f8a-8d93-c2d1-2a08-28a50ca1a466', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the spaghetti refuse to get married. \n\nBecause it was afraid of getting tangled up in a relationship.'}),), interrupts=())

In [20]:
workflow.invoke(None,{"configurable": {"thread_id": "2", "checkpoint_id": "1f12c5b6-dd81-68f7-8000-11b5d143e3da"}})

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married. \n\nBecause it was afraid of getting tangled up in a relationship.',
 'explanation': 'The joke "Why did the spaghetti refuse to get married. Because it was afraid of getting tangled up in a relationship" is a play on words. \n\nThe phrase "tangled up in a relationship" is a common idiomatic expression that means to become deeply involved or entangled in a romantic relationship, often in a way that\'s complicated or difficult to escape. \n\nHowever, in this joke, the phrase takes on a literal meaning as well, as spaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. \n\nThe humor comes from the unexpected twist on the usual meaning of the phrase, and the clever use of the spaghetti\'s physical properties to create a pun that connects the setup and the punchline. The joke relies on a lighthearted and clever play on words to create a sense of surprise and amusement, making i

In [21]:
#Updating State
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f12c5b6-dd81-68f7-8000-11b5d143e3da", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12c5e7-550c-6dc9-8001-fc70dfe01edc'}}